<a href="https://colab.research.google.com/github/mandib96/case-analytics-engineer-2026/blob/camada_raw/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# importando lib e autenticaçao com cloud
import os
import pandas as pd

from google.oauth2 import service_account
from google.cloud import storage, bigquery
from datetime import datetime

service_account_path = '/content/gcp_service_account.json'
credentials = service_account.Credentials.from_service_account_file(service_account_path)

client_bq = bigquery.Client(credentials=credentials, project=credentials.project_id)
client_storage = storage.Client(credentials=credentials, project=credentials.project_id)

#definindo variaveis
bucket_raw = 'case_vendas_boti'
dataset = 'vendas_dataset'

print("Configuração concluida!")

In [ ]:
def ler_arquivos_gcs(bucket_name, prefix='raw_data/'):

    client = storage.Client(credentials=credentials, project=credentials.project_id)
    bucket = client.bucket(bucket_name)

    # Lista todos os arquivos
    blobs = list(bucket.list_blobs())
    csv_blobs = [blob for blob in blobs if blob.name.endswith('.csv')]

    print(f"Arquivos encontrados: {len(csv_blobs)}")

    dataframes = []
    for blob in csv_blobs:
        content = blob.download_as_string()
        df = pd.read_csv(pd.io.common.BytesIO(content))

        # Adiciona metadados
        df['arquivo_origem_gcs'] = blob.name
        df['data_upload_gcs'] = blob.time_created

        dataframes.append(df)
        print(f"  ✓ {blob.name}: {len(df)} linhas | Upload: {blob.time_created}")

    return dataframes

# ler_arquivos_gcs(bucket_raw)

In [ ]:
def consolidar_dataframes(lista_dfs):

    # Empilha dfs, padroniza campos em caixa baixa, adiciona data de processamento

    df_consolidado = pd.concat(lista_dfs, ignore_index=True)
    df_consolidado.columns = df_consolidado.columns.str.lower().str.replace(' ', '_').str.strip()
    df_consolidado['ds'] = pd.to_datetime(datetime.now())

    print(f"Total consolidado: {len(df_consolidado)} linhas")

    return df_consolidado

# lista_dfs = ler_arquivos_gcs(bucket_raw, prefix='raw_data/')
# df_consolidado = consolidar_dataframes(lista_dfs)
# print(df_consolidado.head())


In [ ]:
def criar_tabelas_base():

    # Cria o Dataset
    # dataset_ref = client_bq.dataset(dataset, project=credentials.project_id)
    # new_dataset = bigquery.Dataset(dataset_ref)
    # new_dataset.location = "US" # Define a localização do dataset
    # client_bq.create_dataset(new_dataset, timeout=30)

    # Cria Tabela Raw
    sql_raw = f"""
    CREATE TABLE IF NOT EXISTS `{credentials.project_id}.{dataset}.tbl_raw_vendas` (
        id_marca INT64,
        marca STRING,
        id_linha INT64,
        linha STRING,
        data_venda STRING,
        qtd_venda INT64,
        arquivo_origem_gcs STRING,
        data_upload_gcs TIMESTAMP,
        ds TIMESTAMP NOT NULL
    )
    PARTITION BY DATE(ds)
    """

    try:
        client_bq.query(sql_raw).result()
        print(f"Tabela Raw criada/verificada")

    except Exception as e:
        print(f"Erro ao criar tabela: {e}")
        raise

# criar_tabelas_base()

In [ ]:
def carregar_raw_data(df):

    # Verifica linhas no df
    if df.empty:
        print("df vazio!")
        return

    table_id = f"{credentials.project_id}.{dataset}.tbl_raw_vendas"

    df_raw = df[[
        'id_marca', 'marca', 'id_linha', 'linha', 'data_venda',
        'qtd_venda', 'arquivo_origem_gcs', 'data_upload_gcs', 'ds'
    ]].copy()

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
            time_partitioning=bigquery.TimePartitioning(
            type_=bigquery.TimePartitioningType.DAY,
            field="ds",
        ),
    )

    try:
        job = client_bq.load_table_from_dataframe(
            df_raw,
            table_id,
            job_config=job_config
        )
        job.result()

        print(f"Bronze carregado com sucesso")
        print(f"Job ID: {job.job_id}")
        print(f"Partição: {df_raw['ds'].dt.date.iloc[0]}")

    except Exception as e:
        print(f"Erro ao carregar Bronze: {e}")
        print(f"Dica: Verifique se tipos de dados do DataFrame correspondem ao schema da tabela")
        raise

# carregar_raw_data(df_consolidado)

In [ ]:
def executar_camada_raw():

    print("Iniciando carga na camanda raw")

    try:
        lista_dfs = ler_arquivos_gcs(bucket_raw, prefix='raw_data/')

        if not lista_dfs:
            print("Nenhum arquivo encontrado no path.")
            return

        df_consolidado = consolidar_dataframes(lista_dfs)

        carregar_raw_data(df_consolidado)

        print("Camada raw criada com sucesso!")

        return {
            'status': 'sucesso',
            'arquivos carregados': len(lista_dfs),
            'linhas carregadas': len(df_consolidado),
            'tabela destino': f"{credentials.project_id}.{dataset}.tbl_raw_vendas"
        }

    except Exception as e:
        print(f"ERRO: {str(e)}")
        raise

In [ ]:
if __name__ == "__main__":
    executar_camada_raw()